In [ ]:
%load_ext IPython.extensions.autoreload

In [ ]:
import sys

print(sys.executable)

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Data Validation extension is not supported",
    module="openpyxl"
)

## GENERAL IMPORTS

In [ ]:
from pprint import pprint
from typing import Final

import numpy as np
import pandas as pd

## ``riskquant`` IMPORTS

In [ ]:
from riskquant.config.compile import compile_risk_static_config_from_xlsx
from riskquant.config.resolution import RiskFactorNameAbsToRelMapping
from riskquant.market_data.risk_factors.matrix import RiskFactorMatrix
from riskquant.utils.dates import date_str_to_date
from riskquant.utils.printer import (print_as_list_str, print_df, print_dict, print_success_message, print_wrapped_text_box)

## INPUTS

In [ ]:
# AS OF DATE AS STR
AS_OF_DATE_STR: Final[str] = "2026-01-02"

# Path to the dev data directory
DEV_DATA_DIR: Final[str] = "C:/Users/mharvey/Documents/code/riskquant/data/dev"

# File names of the dev data
DEV_DATA_FILE_NAME_STATIC_CONFIG: Final[str] = "dev_risk_static_data_config_v01.xlsx"
DEV_DATA_FILE_NAME_RISK_FACTOR_LEVELS: Final[str] = "dev_risk_factor_levels_v01.xlsx"
DEV_DATA_FILE_NAME_RISK_FACTOR_LEVELS_SNAPSHOT: Final[str] = "dev_risk_factor_levels_snapshot_v01.xlsx"
DEV_DATA_FILE_NAME_POSITIONS: Final[str] = "dev_positions_v01.xlsx"

## AS OF DATE NORMALISATION

In [ ]:
as_of_date = date_str_to_date(AS_OF_DATE_STR)

## LOAD RISK STATIC DATA CONFIG: ``RiskStaticConfig``

In [ ]:
# Load and compile the config in one step, since we won't need the intermediate dataframes or frames config after compilation.
cfg = compile_risk_static_config_from_xlsx(
    path=DEV_DATA_DIR + "/" + DEV_DATA_FILE_NAME_STATIC_CONFIG
    )

In [ ]:
# Basic identity
print('cfg type:', type(cfg))
print('top-level attributes present:')

# Summary counts
print('num containers:', len(cfg.container_meta_dict))
print('num risk_factors:', len(cfg.risk_factors_dict))

# Samples
print('\nSample risk factor names (first 20):')
pprint(list(cfg.risk_factors_dict.keys())[:20])

print('\nSample container names (first 20):')
pprint(list(cfg.container_meta_dict.keys())[:20])

# Cached lookups
print('\nSample rf_name_to_rf_id (first 20):')
pprint({k: cfg.rf_name_to_rf_id[k] for k in list(cfg.rf_name_to_rf_id)[:20]})

print('\nSample rf_id_to_risk_factor keys (first 20):')
pprint(list(cfg.rf_id_to_risk_factor.keys())[:20])

# Build absolute mapping for as-of
rf_abs_to_rel_mapping = RiskFactorNameAbsToRelMapping.from_config(
    config=cfg, 
    as_of_date=as_of_date
)

# Type. 
print('\nType of rf_abs_to_rel_mapping:', type(rf_abs_to_rel_mapping))

# Display sample of the mapping
print(f'\nSample of absolute to relative name mapping for as_of_date={as_of_date}: \n ')
pprint(list(rf_abs_to_rel_mapping.abs_to_rf_name_and_id.items())[:30])   

In [ ]:
rf_abs_to_rel_mapping.df.to_clipboard()

## LOAD RISK FACTOR LEVELS HISTORY: ``riskquant\market_data\risk_factors\matrix.RiskFactorMatrix``

In [ ]:
from riskquant.market_data.risk_factors.builder_matrix import build_risk_factor_matrix_from_xlsx

In [ ]:
# ---------------------------------------------------------
# LOAD IN HISTORICAL RISK FACTOR LEVELS
# ---------------------------------------------------------

# Print title.
print("\n--- Loading in historical risk factor levels ---")

# Load from XLSX.
rfm = build_risk_factor_matrix_from_xlsx(
    path=DEV_DATA_DIR + "/" + DEV_DATA_FILE_NAME_RISK_FACTOR_LEVELS,
    name_to_id_map=rf_abs_to_rel_mapping.abs_to_rf_id,
)

# Display basic info about the RFM.
print('\nType of RiskFactorMatrix:', type(rfm))
print("\nRFM shape:", rfm.shape)

print("\nRMF Head (first 5 rows, transposed):")
print_df(rfm.head(5).T)
print("\nRMF Tail (last 5 rows, transposed):")
print_df(rfm.tail(5).T)

print("\nPlot of RFM values:")
rfm.plot(figsize=(13, 4))

## LOAD RISK FACTOR LEVELS SNAPSHOT: ``riskquant\market_data\risk_factors\snapshot.RiskFactorLevelSnapshot``

In [ ]:
from riskquant.market_data.risk_factors.builder_snapshot import build_risk_factor_level_snapshot_from_xlsx

In [ ]:
# ---------------------------------------------------------
# LOAD IN HISTORICAL RISK FACTOR LEVELS
# ---------------------------------------------------------

# Print title.
print("\n--- Loading in historical risk factor levels ---")

# Load from XLSX.
rfs = build_risk_factor_level_snapshot_from_xlsx(
    as_of_date=as_of_date,
    path=DEV_DATA_DIR + "/" + DEV_DATA_FILE_NAME_RISK_FACTOR_LEVELS_SNAPSHOT,
    name_to_id_map=rf_abs_to_rel_mapping.abs_to_rf_id,
)

# As df.
print('\nType of RiskFactorLevelSnapshot:', type(rfs))
print("\nSnapshot as dataframe (transposed):")
print_df(rfs.df.T)

# As series.
print("\nSnapshot as series:")
print_df(rfs.sr)

## POSITIONS DATA   

#### IMPORT FROM XLSX

In [ ]:
# Import positions data from XLSX.
positions_df = pd.read_excel(DEV_DATA_DIR + "/" + DEV_DATA_FILE_NAME_POSITIONS)

# Display teh positions data.
print('\nPositions data:')
print(positions_df.to_string(index=False))

#### DATE PARSING: ``riskquant.portfolio.ingest.parser``

In [ ]:
from riskquant.portfolio.ingest.parser import (
    parse_optional_date_series_to_ordinal32 as parse_dates,
)

# Example of parsing a date column from the positions data.
print('\nParsing the "expiry_date" column from the positions data:')
maturity_dates_ordinal32 = parse_dates(
    ser=positions_df["expiry_date"], 
    sentinel=-1,
    col_name="expiry_date",
    ) 
print('Parsed expiry dates (ordinal32):')
print(maturity_dates_ordinal32)

# Example of parsing a date column from the positions data.
print('\nParsing the "settlement_date" column from the positions data:')
maturity_dates_ordinal32 = parse_dates(
    ser=positions_df["settlement_date"], 
    sentinel=-1,
    col_name="settlement_date",
    ) 
print('Parsed settlement dates (ordinal32):')
print(maturity_dates_ordinal32)

#### FX RESOLUTION: ``riskquant.portfolio.ingest.fx_resolver``

In [ ]:
from riskquant.portfolio.ingest.fx_resolver import (
    resolve_fx_rf_ids_inplace as resolve_fx_rf_ids_inplace_fn,
)

base_ccy = positions_df["underlying_ccy"].values
settlement_ccy = positions_df["settlement_ccy"].values 
out_fx_rf_ids = np.empty(base_ccy.shape[0], dtype=np.int32)
rf_name_to_rf_id = rf_abs_to_rel_mapping.rf_name_to_rf_id

resolve_fx_rf_ids_inplace_fn(
    base_ccy=base_ccy,
    settlement_ccy=settlement_ccy, 
    out_fx_rf_ids=out_fx_rf_ids,
    rf_name_to_rf_id=rf_name_to_rf_id,
)

# Display base and settlement currencies alongside the resolved FX RF IDs for verification in a dataframe format.
print('\nBase CCY, Settlement CCY, Resolved FX RF ID:')
verification_df = pd.DataFrame(
    {    
        "base_ccy": base_ccy,     
        "settlement_ccy": settlement_ccy,     
        "resolved_fx_rf_id": out_fx_rf_ids, 
    }
)

# Add FX RF names by mapping IDs back to risk factor names
verification_df["fx_rf_name"] = verification_df["resolved_fx_rf_id"].apply(
    lambda rf_id: cfg.rf_id_to_risk_factor[rf_id].risk_factor_name if rf_id > 0 else "N/A"
)

print(verification_df.to_string(index=False))

In [ ]:
xxx

## CONVERT POSITIONS DATA INTO TRADE RECORDS: ``Iterator[riskquant.portfolio.trades.TradeRecord]``

- There can be different methods used here.
- The key thing is that this layer gets us to an iterator of generic ``TradeRecords`` which are then compiled into a ``CompiledPortfolio``. 
- Getting to an iterator of ``TradeRecords`` is quite critical and may involve different pipelines and different processing of trade and positions data.

In [ ]:
# # Trade records generator
# trade_records_iterator = iter_trades_from_positions_df(
#     positions_df=positions_df,
#     cfg=cfg,
# )


# list_spot_trades_from_positions_df = list(trade_records_iterator)

## COMPILE PORTFOLIO: ``riskquant.portfolio.compiled.CompiledPortfolio``

- The function ``riskquant.portfolio.compiler.compile_portfolio_streaming()`` takes in ``Iterator[riskquant.portfolio.trades.TradeRecord]`` and an instance of a ``riskquant.config.configs.RiskStaticConfig``.

In [ ]:
# # Compile the portfolio
# compiled_portfolio = riskquant.portfolio.compiler.compile_portfolio_streaming(
#     trades=trade_records_iterator, 
#     cfg=cfg,
# )

# # Delete trade_records_iterator (releases iterator + captured arrays)
# del trade_records_iterator

In [ ]:
# proxy_resolution = riskquant.simlab.resolve.proxy.resolve_required_factor_ids_to_market_data_factor_ids(
#     cfg=cfg, 
#     required_factor_ids=compiled_portfolio.factor_ids_sorted_unique, 
#     market_data_factor_ids=rfm.factor_ids, 
# )

In [ ]:
# proxy_res_audit = riskquant.simlab.resolve.proxy_audit.ProxyResolutionAudit(cfg=cfg, proxy_resolution=proxy_resolution)
# proxy_res_audit.df